In [88]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pickle


from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.metrics import mutual_info_score, roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier


from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant

import xgboost as xgb


import warnings
warnings.filterwarnings('ignore')


In [19]:
# Read data
red_wine = pd.read_csv('data/winequality-red.csv', sep=';')
white_wine = pd.read_csv('data/winequality-white.csv', sep=';')
# Type variable
red_wine['is_red'] = 1
white_wine['is_red'] = 0
# Concat
df = pd.concat([red_wine, white_wine], ignore_index=True, sort=False)
# Format col names
df.columns = [col.lower().replace(' ', '_') for col in df.columns]
# Target colum
df['label'] = (df['quality'] >= 7).astype(int)
del df['quality']
df

,fixed_acidity,volatile_acidity,citric_acid,residual_sugar,chlorides,free_sulfur_dioxide,total_sulfur_dioxide,density,ph,sulphates,alcohol,is_red,label
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.99780,3.51,0.56,9.4,1,0
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.99680,3.20,0.68,9.8,1,0
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.99700,3.26,0.65,9.8,1,0
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.99800,3.16,0.58,9.8,1,0
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.99780,3.51,0.56,9.4,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
6492,6.2,0.21,0.29,1.6,0.039,24.0,92.0,0.99114,3.27,0.50,11.2,0,0
6493,6.6,0.32,0.36,8.0,0.047,57.0,168.0,0.99490,3.15,0.46,9.6,0,0
6494,6.5,0.24,0.19,1.2,0.041,30.0,111.0,0.99254,2.99,0.46,9.4,0,0
6495,5.5,0.29,0.30,1.1,0.022,20.0,110.0,0.98869,3.34,0.38,12.8,0,1


## **EDA**

In [ ]:
# Nulls
df.isnull().sum()

fixed_acidity           0
volatile_acidity        0
citric_acid             0
residual_sugar          0
chlorides               0
free_sulfur_dioxide     0
total_sulfur_dioxide    0
density                 0
ph                      0
sulphates               0
alcohol                 0
is_red                  0
label                   0
dtype: int64

In [ ]:
# Statistics
df.describe()

,fixed_acidity,volatile_acidity,citric_acid,residual_sugar,chlorides,free_sulfur_dioxide,total_sulfur_dioxide,density,ph,sulphates,alcohol,is_red,label
count,6497.000000,6497.000000,6497.000000,6497.000000,6497.000000,6497.000000,6497.000000,6497.000000,6497.000000,6497.000000,6497.000000,6497.000000,6497.000000
mean,7.215307,0.339666,0.318633,5.443235,0.056034,30.525319,115.744574,0.994697,3.218501,0.531268,10.491801,0.246114,0.196552
std,1.296434,0.164636,0.145318,4.757804,0.035034,17.749400,56.521855,0.002999,0.160787,0.148806,1.192712,0.430779,0.397421
min,3.800000,0.080000,0.000000,0.600000,0.009000,1.000000,6.000000,0.987110,2.720000,0.220000,8.000000,0.000000,0.000000
25%,6.400000,0.230000,0.250000,1.800000,0.038000,17.000000,77.000000,0.992340,3.110000,0.430000,9.500000,0.000000,0.000000
50%,7.000000,0.290000,0.310000,3.000000,0.047000,29.000000,118.000000,0.994890,3.210000,0.510000,10.300000,0.000000,0.000000
75%,7.700000,0.400000,0.390000,8.100000,0.065000,41.000000,156.000000,0.996990,3.320000,0.600000,11.300000,0.000000,0.000000
max,15.900000,1.580000,1.660000,65.800000,0.611000,289.000000,440.000000,1.038980,4.010000,2.000000,14.900000,1.000000,1.000000


In [ ]:
# Label distribution
df['label'].value_counts(normalize=True)

label
0    0.803448
1    0.196552
Name: proportion, dtype: float64

## **Splits**

In [65]:
full_train, test = train_test_split(df, test_size=0.10, random_state=0, stratify=df['label'])
train, val = train_test_split(full_train, test_size=0.20, random_state=0, stratify=full_train['label'])
X_train = train.drop(columns='label')
y_train = train['label']
X_val = val.drop(columns='label')
y_val = val['label']
X_test = test.drop(columns='label')
y_test = test['label']
print(f'Numero registros train: {len(train)}')
print(f'Distribucion y train: {y_train.value_counts(normalize=True)}')
print(f'Numero registros val: {len(val)}')
print(f'Distribucion y val: {y_val.value_counts(normalize=True)}')
print(f'Numero registros test: {len(test)}')
print(f'Distribucion y test: {y_test.value_counts(normalize=True)}')

Numero registros train: 4677
Distribucion y train: label
0    0.803507
1    0.196493
Name: proportion, dtype: float64
Numero registros val: 1170
Distribucion y val: label
0    0.803419
1    0.196581
Name: proportion, dtype: float64
Numero registros test: 650
Distribucion y test: label
0    0.803077
1    0.196923
Name: proportion, dtype: float64


## **Feature Importance**

In [66]:
X_train.columns

Index(['fixed_acidity', 'volatile_acidity', 'citric_acid', 'residual_sugar',
       'chlorides', 'free_sulfur_dioxide', 'total_sulfur_dioxide', 'density',
       'ph', 'sulphates', 'alcohol', 'is_red'],
      dtype='str')

In [67]:
num_cols = ['fixed_acidity', 'volatile_acidity', 'citric_acid', 'residual_sugar',
            'chlorides', 'free_sulfur_dioxide', 'total_sulfur_dioxide', 'density',
            'ph', 'sulphates', 'alcohol']
correlations = X_train[num_cols].corrwith(y_train).sort_values()
correlations

density                -0.275351
chlorides              -0.160470
volatile_acidity       -0.150496
residual_sugar         -0.058456
fixed_acidity          -0.050275
total_sulfur_dioxide   -0.046406
free_sulfur_dioxide     0.019127
ph                      0.022872
sulphates               0.031208
citric_acid             0.052738
alcohol                 0.391533
dtype: float64

In [68]:
def calculate_mi(series):
    return mutual_info_score(series, train['label'])

cat_cols = ['is_red']
df_mi = train[cat_cols].apply(calculate_mi)
df_mi = df_mi.sort_values(ascending=False).to_frame(name='MI_score')
df_mi

,MI_score
is_red,0.004329


In [69]:
pre_selection = ['density', 'chlorides', 'volatile_acidity', 'alcohol']
X_train[pre_selection].corr()

,density,chlorides,volatile_acidity,alcohol
density,1.000000,0.369614,0.273012,-0.702480
chlorides,0.369614,1.000000,0.382537,-0.255927
volatile_acidity,0.273012,0.382537,1.000000,-0.036848
alcohol,-0.702480,-0.255927,-0.036848,1.000000


In [70]:
def calcular_vif(X):
    # statsmodels requiere que añadamos una constante para calcular el intercepto
    X_con_constante = add_constant(X)
    
    vif_data = pd.DataFrame()
    vif_data["Variable"] = X_con_constante.columns
    
    # Calculamos el VIF para cada variable
    vif_data["VIF"] = [variance_inflation_factor(X_con_constante.values, i) 
                       for i in range(X_con_constante.shape[1])]
    
    # Filtramos la constante para ver solo nuestras features y ordenamos
    vif_data = vif_data[vif_data['Variable'] != 'const'].sort_values(by="VIF", ascending=False)
    return vif_data

# Ejecutamos la función sobre nuestras variables numéricas
vif_resultados = calcular_vif(X_train[pre_selection])
print(vif_resultados)

           Variable       VIF
1           density  2.307276
4           alcohol  2.091359
2         chlorides  1.292392
3  volatile_acidity  1.270737


## **Modelamiento**

### Logistic Regression

In [80]:
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('logreg', LogisticRegression(random_state=42, max_iter=10000))
])

param_grid = [
    {'logreg__penalty': ['l2'], 'logreg__C': [0.001, 0.01, 0.1, 1, 10, 100], 'logreg__solver':['lbfgs', 'liblinear']}, 
    {'logreg__penalty': ['l1'], 'logreg__C': [0.001, 0.01, 0.1, 1, 10, 100], 'logreg__solver':['liblinear']}, 
    {'logreg__penalty': ['elasticnet'], 'logreg__C': [0.001, 0.01, 0.1, 1, 10, 100], 
     'logreg__l1_ratio': [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9], 'logreg__solver':['saga']}
]

cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid_search = GridSearchCV(
    estimator = pipeline,
    param_grid = param_grid,
    scoring = 'roc_auc',
    cv = cv_strategy,
    refit = True,
    n_jobs = -1,
    verbose = 1
)
grid_search.fit(X_train[pre_selection], y_train)
model_rl = grid_search.best_estimator_
best_params_clean = {k.replace('logreg__', ''): v for k, v in grid_search.best_params_.items()}
print(f'Params: {best_params_clean}')

y_predict_train = model_rl.predict_proba(X_train[pre_selection])[:, 1]
y_predict_val = model_rl.predict_proba(X_val[pre_selection])[:, 1]
roc_train_rl = roc_auc_score(y_train, y_predict_train)
roc_val_rl = roc_auc_score(y_val, y_predict_val)
print(f'ROC train {roc_train_rl:.4f}, ROC test {roc_val_rl:.4f}')

Fitting 5 folds for each of 72 candidates, totalling 360 fits
Params: {'C': 100, 'penalty': 'l1', 'solver': 'liblinear'}
ROC train 0.8000, ROC test 0.7901


### Random Forest

In [85]:
param_grid_rf = {
    'n_estimators': [300, 500],          # Cantidad de árboles en el bosque
    'max_depth': [5, 7],           # Control principal de complejidad
    'min_samples_leaf': [10, 15],           # Evita hojas con muy pocos registros (ruido)
    'min_samples_split': [20, 25]           # Registros mínimos para permitir una nueva partición
}

cv_strategy_rf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid_search_rf = GridSearchCV(
    estimator = RandomForestClassifier(random_state=42, n_jobs=-1),
    param_grid = param_grid_rf,
    scoring = 'roc_auc',
    cv = cv_strategy_rf,
    refit = True,
    n_jobs = -1,
    verbose = 1
)

features_rf = X_train.columns
grid_search_rf.fit(X_train[features_rf], y_train)
model_rf = grid_search_rf.best_estimator_

print(f'Mejores parámetros RF: {grid_search_rf.best_params_}')

y_predict_train_rf = model_rf.predict_proba(X_train[features_rf])[:, 1]
y_predict_val_rf = model_rf.predict_proba(X_val[features_rf])[:, 1]

roc_train_rf = roc_auc_score(y_train, y_predict_train_rf)
roc_val_rf = roc_auc_score(y_val, y_predict_val_rf)

print(f'ROC train RF: {roc_train_rf:.4f}, ROC val RF: {roc_val_rf:.4f}')

Fitting 5 folds for each of 16 candidates, totalling 80 fits
Mejores parámetros RF: {'max_depth': 7, 'min_samples_leaf': 10, 'min_samples_split': 20, 'n_estimators': 300}
ROC train RF: 0.9186, ROC val RF: 0.8576


### XGBoost

In [87]:
dtrain = xgb.DMatrix(X_train[features_rf], label=y_train)
dval = xgb.DMatrix(X_val[features_rf], label=y_val)

# 2. Configurar hiperparámetros con fuerte regularización
xgb_params = {
    'objective': 'binary:logistic',
    'eval_metric': 'auc',
    'eta': 0.05,                # Tasa de aprendizaje baja (aprende despacio)
    'max_depth': 6,             # Profundidad moderada
    'min_child_weight': 5,      # Equivalente a min_samples_leaf, restringe ruido
    'subsample': 0.8,           # Inyecta aleatoriedad: usa 80% de datos por árbol
    'colsample_bytree': 0.8,    # Inyecta aleatoriedad: usa 80% de variables por árbol
    'lambda': 1,                # Regularización L2 (Ridge)
    'alpha': 0.1,               # Regularización L1 (Lasso)
    'seed': 42
}

# 3. Lista de evaluación para monitorear train y val simultáneamente en cada iteración
watchlist = [(dtrain, 'train'), (dval, 'val')]

# 4. Entrenamiento con Early Stopping
# Se detendrá automáticamente si el AUC de validación no mejora durante 50 árboles consecutivos
model_xgb = xgb.train(
    xgb_params,
    dtrain,
    num_boost_round=1000,
    evals=watchlist,
    early_stopping_rounds=50,
    verbose_eval=50
)

# 5. Evaluación final del mejor modelo encontrado
y_predict_train_xgb = model_xgb.predict(dtrain)
y_predict_val_xgb = model_xgb.predict(dval)

roc_train_xgb = roc_auc_score(y_train, y_predict_train_xgb)
roc_val_xgb = roc_auc_score(y_val, y_predict_val_xgb)

print(f'\nROC train XGBoost: {roc_train_xgb:.4f}')
print(f'ROC val XGBoost: {roc_val_xgb:.4f}')

[0]	train-auc:0.86398	val-auc:0.79923
[50]	train-auc:0.93660	val-auc:0.86844
[100]	train-auc:0.95265	val-auc:0.87655
[150]	train-auc:0.96544	val-auc:0.88245
[200]	train-auc:0.97510	val-auc:0.88606
[250]	train-auc:0.98238	val-auc:0.88714
[300]	train-auc:0.98816	val-auc:0.88877
[350]	train-auc:0.99172	val-auc:0.88971
[400]	train-auc:0.99409	val-auc:0.89038
[450]	train-auc:0.99617	val-auc:0.89074
[500]	train-auc:0.99757	val-auc:0.89127
[550]	train-auc:0.99846	val-auc:0.89181
[600]	train-auc:0.99896	val-auc:0.89287
[650]	train-auc:0.99938	val-auc:0.89251
[671]	train-auc:0.99949	val-auc:0.89220

ROC train XGBoost: 0.9995
ROC val XGBoost: 0.8922


### Modelo final

In [89]:
X_full_train = pd.concat([X_train[features_rf], X_val[features_rf]])
y_full_train = pd.concat([y_train, y_val])

rf_final_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=7,
    min_samples_leaf=10,
    min_samples_split=20,
    random_state=42,
    n_jobs=-1
)

rf_final_model.fit(X_full_train, y_full_train)

y_predict_test = rf_final_model.predict_proba(X_test[features_rf])[:, 1]
roc_test = roc_auc_score(y_test, y_predict_test)

print(f'ROC definitivo en TEST: {roc_test:.4f}')

# 5. Exportar el modelo serializado
output_file = 'model_rf.bin'
with open(output_file, 'wb') as f_out:
    # Si hubieras usado DictVectorizer o StandardScaler, también se empaquetarían aquí
    pickle.dump(rf_final_model, f_out)

print(f'Modelo exportado exitosamente a {output_file}')

ROC definitivo en TEST: 0.8669
Modelo exportado exitosamente a model_rf.bin
